<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/ipca/IPCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IPCA — acumulado em 12 meses

Este notebook é o motor de coleta e publicação da série `ipca-acum-12m`.

- **Fonte:** IBGE, via API de Dados Abertos do Banco Central (SGS), série **13522** — “IPCA, variação acumulada em 12 meses (%)”.
- **Endpoint:** `https://api.bcb.gov.br/dados/serie/bcdata.sgs.13522/dados?formato=json&dataInicial=01/01/1995&dataFinal=<hoje>`
- **Definição:** variação percentual do IPCA acumulada nos últimos 12 meses (taxa efetiva do período), unidade `percentual` (`%`, 2 casas), `eixo: percentual`, `medida: acumulado-12m`, granularidade `mensal`.
- **Fórmula:** nenhuma. O cálculo é definido pela fonte; o notebook apenas busca a série oficial, filtra os meses fechados e formata.
- **Período publicado:** começa em `1995-01-01` e termina no último mês fechado disponível.
- **Arquivo gerado:** `indicadores/ipca/ipca-acum-12m.json`, com `versao: 1` e `id: ipca-acum-12m`.

## Decisão metodológica

A série publicada é a oficial **SGS 13522**, sem recalcular a composição a partir da SGS 433 (variação mensal); a 433 não participa da saída. A 13522 já traz o acumulado de 12 meses, então cobre os **11 pontos de 1995-01 a 1995-11** que a composição da 433 não alcançaria — ela só fecharia a primeira janela em `1995-12` — e vai até o último mês fechado.


## Coleta da série oficial

A leitura é feita desde `1995-01-01`, sem atualização incremental e sem ler o JSON já publicado para decidir o período.


In [1]:
from datetime import date, datetime
from pathlib import Path
import json
import os
import time

import pandas as pd
import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()

def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name
PASTA = RAIZ / "indicadores" / "ipca" if (RAIZ / "indicadores" / "ipca").is_dir() else RAIZ
ARQUIVO = PASTA / "ipca-acum-12m.json"

SERIE = 13522
URL = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{SERIE}/dados"
INICIO = date(1995, 1, 1)
HOJE = date.today()

for tentativa in range(4):
    try:
        resposta = requests.get(
            URL,
            params={
                "formato": "json",
                "dataInicial": INICIO.strftime("%d/%m/%Y"),
                "dataFinal": HOJE.strftime("%d/%m/%Y"),
            },
            timeout=60,
        )
        resposta.raise_for_status()
        bruto = resposta.json()
        break
    except (requests.RequestException, ValueError):
        if tentativa == 3:
            raise
        time.sleep(2 ** tentativa)

if not bruto:
    raise ValueError(f"a série SGS {SERIE} não retornou dados")

df = pd.DataFrame(
    {
        "data": pd.to_datetime(
            [datetime.strptime(item["data"], "%d/%m/%Y") for item in bruto]
        ),
        "valor": [float(item["valor"]) for item in bruto],
    }
).sort_values("data", ignore_index=True)

print(f"Série SGS {SERIE}: {len(df)} leituras")
print(f"Período bruto: {df['data'].min():%Y-%m-%d} a {df['data'].max():%Y-%m-%d}")
print(f"Arquivo de saída: {relativo(ARQUIVO)}")
print("\nPrimeiras 5 leituras:")
print(df.head().to_string(index=False))
print("\nÚltimas 5 leituras:")
print(df.tail().to_string(index=False))


Série SGS 13522: 381 leituras
Período bruto: 1995-01-01 a 2026-09-01
Arquivo de saída: indicadores/ipca/ipca-acum-12m.json

Primeiras 5 leituras:
      data  valor
1995-01-01 631.54
1995-02-01 426.83
1995-03-01 274.78
1995-04-01 169.05
1995-05-01  91.79

Últimas 5 leituras:
      data  valor
2026-05-01   4.72
2026-06-01   4.64
2026-07-01   4.44
2026-08-01   4.22
2026-09-01   4.58


## Filtragem dos meses fechados

O mês corrente é descartado porque ainda está aberto. A fonte devolve as datas no primeiro dia de cada mês; essa convenção é preservada.


In [2]:
primeiro_dia_mes_corrente = pd.Timestamp(HOJE.replace(day=1))
fechados = df[df["data"] < primeiro_dia_mes_corrente].copy()

if fechados.empty:
    raise ValueError("a fonte não retornou nenhum mês fechado")

if fechados["data"].dt.to_period("M").duplicated().any():
    raise ValueError("a fonte retornou meses repetidos")

print(f"Mês corrente excluído: {HOJE:%Y-%m}")
print(f"Meses fechados publicados: {len(fechados)}")
print(
    f"Primeiro: {fechados['data'].iloc[0]:%Y-%m-%d} "
    f"({fechados['valor'].iloc[0]:.2f}%)"
)
print(
    f"Último:   {fechados['data'].iloc[-1]:%Y-%m-%d} "
    f"({fechados['valor'].iloc[-1]:.2f}%)"
)


Mês corrente excluído: 2026-11
Meses fechados publicados: 381
Primeiro: 1995-01-01 (631.54%)
Último:   2026-09-01 (4.58%)


## Escrita do JSON canônico

Os pontos são gravados como `{"data": "YYYY-MM-DD", "valor": ...}`, em ordem ascendente e sem campos duplicados.


In [3]:
pontos = [
    {"data": data.strftime("%Y-%m-%d"), "valor": float(valor)}
    for data, valor in zip(fechados["data"], fechados["valor"])
]

serie = {
    "versao": 1,
    "id": "ipca-acum-12m",
    "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 2},
    "eixo": "percentual",
    "medida": "acumulado-12m",
    "granularidade": "mensal",
    "fonte": "IBGE",
    "urlFonte": f"{URL}?formato=json",
    "dados": pontos,
}

with ARQUIVO.open("w", encoding="utf-8") as arquivo:
    json.dump(serie, arquivo, ensure_ascii=False, indent=2)

print(f"{len(pontos)} pontos escritos em {ARQUIVO.name}")
print(f"Primeiro ponto: {pontos[0]}")
print(f"Último ponto:   {pontos[-1]}")


381 pontos escritos em ipca-acum-12m.json
Primeiro ponto: {'data': '1995-01-01', 'valor': 631.54}
Último ponto:   {'data': '2026-09-01', 'valor': 4.58}


## Validação inline

A célula final falha se o contrato, a unicidade dos meses, a ordem, a finitude dos valores ou a igualdade com a fonte forem violados. Também confere os números citados em `docs/metodologia.md`.


In [4]:
import math

CHAVES = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
assert set(serie) == CHAVES, sorted(set(serie) ^ CHAVES)
assert serie["versao"] == 1, serie["versao"]
assert serie["id"] == "ipca-acum-12m", serie["id"]
assert serie["unidade"] == {"tipo": "percentual", "simbolo": "%", "casas": 2}, serie["unidade"]
assert serie["eixo"] == "percentual", serie["eixo"]
assert serie["medida"] == "acumulado-12m", serie["medida"]
assert serie["granularidade"] == "mensal", serie["granularidade"]
assert serie["fonte"] == "IBGE", serie["fonte"]
assert serie["urlFonte"] == f"{URL}?formato=json", serie["urlFonte"]

datas = [ponto["data"] for ponto in serie["dados"]]
meses = [data[:7] for data in datas]

assert datas, "série vazia"
assert datas == sorted(datas), "datas fora de ordem"
assert len(meses) == len(set(meses)), "mês repetido"
assert datas[0] == "1995-01-01", datas[0]
assert datas[-1][:7] < f"{HOJE:%Y-%m}", datas[-1]
assert len(datas) == len(fechados), (len(datas), len(fechados))

for ponto in serie["dados"]:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto

esperado_da_fonte = [
    {"data": data.strftime("%Y-%m-%d"), "valor": float(valor)}
    for data, valor in zip(fechados["data"], fechados["valor"])
]
assert serie["dados"] == esperado_da_fonte, "valor publicado difere da fonte"

por_data = {ponto["data"]: ponto["valor"] for ponto in serie["dados"]}
assert por_data["2025-06-01"] == 5.35, por_data["2025-06-01"]
assert por_data["2002-12-01"] == 12.53, por_data["2002-12-01"]
assert por_data["2003-02-01"] == 15.85, por_data["2003-02-01"]
assert datas[:12] == [f"1995-{mes:02d}-01" for mes in range(1, 13)], datas[:12]

with ARQUIVO.open(encoding="utf-8") as arquivo:
    salvo = json.load(arquivo)
assert salvo == serie, "arquivo gravado difere do objeto validado"

print(
    f"ok: {len(datas)} meses únicos, {datas[0]} a {datas[-1]}, "
    f"valores idênticos à SGS 13522"
)


ok: 381 meses únicos, 1995-01-01 a 2026-09-01, valores idênticos à SGS 13522
